# What makes a basketball team win?

The first time I watched Shai Gilgeous-Alexander play on TV, I was treated to the statistic that it was his `20`th consecutive game in which he had scored `20`+ points (against the Suns on `November 28, 2025`). His stellar season performance did not taper off into playoffs; he posted eye-popping averages of `27.6` points and `7.9` assists per game.

Yet, Oklahoma City failed to repeat as champions, thanks in part to a San Antonio defense spearheaded by Victor Wembanyama.

That got me wondering:

1. *How much do individual box-score stats actually contribute to winning basketball games?*
2. *Do playoffs have a different winning formula from the regular season?*

Let's investigate.

---
## Methodology

What I'm solving for can be described in 2 ways:
1. If we know how often a team shoots, rebounds, passes, steals, blocks shots, and turns the ball over, can we estimate how likely the team is to win?

2. Formally, `win probability per match (%) = (a * box_score_metric_1) + (b * box_score_metric_2) + ... + intercept`

To conduct this experiment, I am  using a type of regression model called Elastic Net.

I chose this regression in particular for two reasons:
1. Many basketball statistics are related to one another.

    - Teams that score more points (`PTS`) often record more assists (`AST`).

    - Teams that shoot efficiently (`FG_PCT`) tend to win more games (`WIN_PCT`).


2. I don't know beforehand which statistics matter most.

    - Some statistics may provide unique information.

    - Others may simply tell us the same story in a different way.

Elastic Net helps solve this problem by automatically identifying the most useful metrics while reducing the influence of redundant ones.

___


## Running the Experiment

Let's jump into the code.

I'm using 2 datasets:
- 2021-2026 regular season. `5` seasons, `30` teams, and `82` games/season/team - a healthy sample size for this regression.
- 2021-2026 playoffs.

In [15]:
from pathlib import Path
import time

import pandas as pd
from nba_api.stats.endpoints import leaguegamelog
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler

'''
The notebook keeps raw game logs locally, then builds the team-season table from those games.
'''

# Seasons included in the analysis.
seasons = ["2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]

# Local CSV files keep NBA.com downloads from running every time.
DATA_DIR = Path("data/nba")
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "regular_season": DATA_DIR / "nba_team_games_regular_season.csv",
    "playoffs": DATA_DIR / "nba_team_games_playoffs.csv",
}

Let's fetch the data from the `nba_api` library and print column headers for the regular season to see what we're working with.

In [28]:
# Gets one row per team-game from NBA.com for each season.
def fetch_games(season_type):
    all_seasons = []

    for season in seasons:
        print(f"Fetching {season} {season_type} games...")

        games = leaguegamelog.LeagueGameLog(
            player_or_team_abbreviation="T",
            season=season,
            season_type_all_star=season_type,
            timeout=60,
        ).get_data_frames()[0]

        games["SEASON"] = season
        games["SEASON_TYPE"] = season_type
        all_seasons.append(games)
        time.sleep(1)

    return pd.concat(all_seasons, ignore_index=True)


# Uses the local CSV if it exists; otherwise fetches the data once and saves it.
def load_or_fetch(path, fetch_fn):
    if path.exists():
        print(f"Loading local file: {path}")
        return pd.read_csv(path)

    data = fetch_fn()
    data.to_csv(path, index=False)
    return data


# Turns one row per team-game into one row per team-season.
def summarize_team_seasons(games):
    games = games.copy()

    # A team's blocks against are the opponent's blocks in the same game.
    opponent_totals = games.groupby("GAME_ID")[["BLK", "PF"]].transform("sum")
    games["BLKA"] = opponent_totals["BLK"] - games["BLK"]
    games["PFD"] = opponent_totals["PF"] - games["PF"]

    # Wins come directly from the game result column.
    games["WIN"] = games["WL"].eq("W").astype(int)

    team_columns = ["SEASON", "SEASON_TYPE", "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME"]
    stats_to_sum = [
        "MIN", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "BLKA",
        "TOV", "PF", "PFD", "PTS", "PLUS_MINUS",
    ]

    record = games.groupby(team_columns, as_index=False).agg(
        GP=("GAME_ID", "count"),
        W=("WIN", "sum"),
    )
    totals = games.groupby(team_columns, as_index=False)[stats_to_sum].sum()
    summary = record.merge(totals, on=team_columns)

    # Rebuild the summary stats from the game rows.
    summary["L"] = summary["GP"] - summary["W"]
    summary["W_PCT"] = summary["W"] / summary["GP"]
    summary["FG_PCT"] = summary["FGM"] / summary["FGA"]
    summary["FG3_PCT"] = summary["FG3M"] / summary["FG3A"]
    summary["FT_PCT"] = summary["FTM"] / summary["FTA"]

    per_game_cols = [
    "FGA", "FG3A", "REB", "AST", "STL", "BLK", "BLKA",
    "TOV", "PF", "PFD", "PTS", "PLUS_MINUS",
    "OREB", "DREB", "FGM", "FG3M", "FTM", "FTA",
]

    for col in per_game_cols:
        summary[f"{col}_PG"] = summary[col] / summary["GP"]

    return summary

df_regular_season = load_or_fetch(
    files["regular_season"],
    lambda: fetch_games("Regular Season"),
)

df_playoffs = load_or_fetch(
    files["playoffs"],
    lambda: fetch_games("Playoffs"),
)

print("Regular-season team-game rows:", len(df_regular_season))
print("Playoff team-game rows:", len(df_playoffs))
df_model_source = summarize_team_seasons(df_regular_season)
df_playoffs_model_source = summarize_team_seasons(df_playoffs)
print("Regular-season team-season cols:\n", df_model_source.columns)

Loading local file: data/nba/nba_team_games_regular_season.csv
Loading local file: data/nba/nba_team_games_playoffs.csv
Regular-season team-game rows: 12300
Playoff team-game rows: 844
Regular-season team-season cols:
 Index(['SEASON', 'SEASON_TYPE', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME',
       'GP', 'W', 'MIN', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB',
       'DREB', 'REB', 'AST', 'STL', 'BLK', 'BLKA', 'TOV', 'PF', 'PFD', 'PTS',
       'PLUS_MINUS', 'L', 'W_PCT', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'FGA_PG',
       'FG3A_PG', 'REB_PG', 'AST_PG', 'STL_PG', 'BLK_PG', 'BLKA_PG', 'TOV_PG',
       'PF_PG', 'PFD_PG', 'PTS_PG', 'PLUS_MINUS_PG', 'OREB_PG', 'DREB_PG',
       'FGM_PG', 'FG3M_PG', 'FTM_PG', 'FTA_PG'],
      dtype='str')


Let's clean up the regular season dataset by defining only what the model should see.


In [22]:
target = "W_PCT"

# These columns either identify the team, are the answer, or are too directly tied to the answer.
model_features = [
    "FGA_PG", "FG3A_PG", "FTA_PG", "OREB_PG", "DREB_PG", "AST_PG",
    "STL_PG", "BLK_PG", "BLKA_PG", "TOV_PG",
    "FG_PCT", "FG3_PCT", "FT_PCT",
]

y = df_model_source[target]
X = df_model_source[model_features]
df_model = pd.concat([y, X], axis=1).dropna()

print("Features used by the model:")
print(list(X.columns))


Features used by the model:
['FGA_PG', 'FG3A_PG', 'FTA_PG', 'OREB_PG', 'DREB_PG', 'AST_PG', 'STL_PG', 'BLK_PG', 'BLKA_PG', 'TOV_PG', 'FG_PCT', 'FG3_PCT', 'FT_PCT']


Now, we have defined that the regression will use per-game shooting volume, rebounding, passing, defensive activity, turnovers, and shooting efficiency to estimate team win percentage.

Next, we will build the model.

In [ ]:
y = df_model["W_PCT"]
X = df_model.drop(columns=["W_PCT"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    cv=5,
    random_state=42,
    max_iter=100000,
)
model.fit(X_scaled, y)

I want to bring attention to `X_scaled` - this step scales the metrics to be standardized versions of themselves.

Elastic Net regression is sensitive to variable size, and the metrics live on different scales: `FG-PCT` could be around `0.4` and `FGA-PG` could be `90`.

Scaling puts every feature on the same footing.

___

Now that the regression has been run, let's now identify the exact features and their coefficients.

In [ ]:
raw_coefficients = model.coef_ / scaler.scale_
raw_intercept = model.intercept_ - (raw_coefficients * scaler.mean_).sum()

formula_df = pd.DataFrame({
    "Feature": X.columns,
    "Raw_Coefficient": raw_coefficients,
})

formula_df["Win_Pct_Points_Per_Unit"] = formula_df["Raw_Coefficient"] * 100
formula_df["Abs_Coeff"] = formula_df["Raw_Coefficient"].abs()

important_formula_terms = (
    formula_df[formula_df["Raw_Coefficient"] != 0]
    .sort_values("Abs_Coeff", ascending=False)
)

important_formula_terms["Raw_Coefficient"] = important_formula_terms["Raw_Coefficient"].round(4)

print("--- Regression using regular season data ---")
print("Formula intercept:", round(raw_intercept, 4))
print(important_formula_terms[["Feature", "Raw_Coefficient"]])

--- Regression using regular season data ---
Formula intercept: -1.9543
    Feature  Raw_Coefficient
10   FG_PCT           4.0053
11  FG3_PCT           1.4594
12   FT_PCT           0.4010
6    STL_PG           0.0510
9    TOV_PG          -0.0480
3   OREB_PG           0.0470
4   DREB_PG           0.0366
8   BLKA_PG          -0.0313
0    FGA_PG          -0.0201
7    BLK_PG           0.0106
1   FG3A_PG           0.0067
5    AST_PG          -0.0033
2    FTA_PG           0.0011


Onto the final step; let's repeat the experiment with playoffs data.

In [34]:
target = "W_PCT"

# These columns either identify the team, are the answer, or are too directly tied to the answer.
model_features = [
    "FGA_PG", "FG3A_PG", "FTA_PG", "OREB_PG", "DREB_PG", "AST_PG",
    "STL_PG", "BLK_PG", "BLKA_PG", "TOV_PG",
    "FG_PCT", "FG3_PCT", "FT_PCT",
]

y = df_playoffs_model_source[target]
X = df_playoffs_model_source[model_features]
df_model = pd.concat([y, X], axis=1).dropna()

y = df_model["W_PCT"]
X = df_model.drop(columns=["W_PCT"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    cv=5,
    random_state=42,
    max_iter=100000,
)
model.fit(X_scaled, y)

raw_coefficients = model.coef_ / scaler.scale_
raw_intercept = model.intercept_ - (raw_coefficients * scaler.mean_).sum()

formula_df = pd.DataFrame({
    "Feature": X.columns,
    "Raw_Coefficient": raw_coefficients,
})

formula_df["Win_Pct_Points_Per_Unit"] = formula_df["Raw_Coefficient"] * 100
formula_df["Abs_Coeff"] = formula_df["Raw_Coefficient"].abs()

important_formula_terms = (
    formula_df[formula_df["Raw_Coefficient"] != 0]
    .sort_values("Abs_Coeff", ascending=False)
)

important_formula_terms["Raw_Coefficient"] = important_formula_terms["Raw_Coefficient"].round(4)

print("--- Regression using playoffs data ---")
print("Formula intercept:", round(raw_intercept, 4))
print(important_formula_terms[["Feature", "Raw_Coefficient"]])

--- Regression using playoffs data ---
Formula intercept: -1.581
    Feature  Raw_Coefficient
10   FG_PCT           2.1975
11  FG3_PCT           0.8077
12   FT_PCT           0.3102
6    STL_PG           0.0544
9    TOV_PG          -0.0417
4   DREB_PG           0.0363
3   OREB_PG           0.0296
0    FGA_PG          -0.0134
7    BLK_PG           0.0076
1   FG3A_PG           0.0054
5    AST_PG           0.0052
8   BLKA_PG           0.0005


## Findings

Across both regular season data and playoffs data, the biggest finding is that ***shooting efficiency matters most***. `FG_PCT` and `FG3_PCT` are the strongest factors in determining win rate.

The per-game stats also make a lot of sense:
- More steals -> higher expected win percentage.
- More turnovers -> lower expected win percentage.
- More offensive/defensive rebounds -> higher expected win percentage.

The main difference is that ***shooting efficiency matters much more in the regular-season model***.
```
FG_PCT
Regular season: 4.0053
Playoffs: 2.1975

FG3_PCT
Regular season: 1.4594
Playoffs: 0.8077
```
Blocks against practically disappear in the playoff model:
```
BLK_PG
Regular season: +0.0106
Playoffs: +0.0076

BLKA_PG
Regular season: -0.0313
Playoffs: +0.0005
```
Getting blocked hurts about *3x more in the regular season* while *barely registering as a factor during the playoffs*.


There are a few assumptions we relied on to derive these:
- Historical box score stats contain useful information to determine win probability.
- The regression methodology of Elastic Net regularization is effective at identifying the most relevant variables that determine win probability.

# Conclusion

The regular winning formula isn't just about scoring more. It is about ***shooting efficiently, creating extra possessions, protecting the ball, and ending opponents' possessions.***

```
Win Probability (%) = 
(shooting stats) 4.0053 * FG_PCT + 1.4594 * FG3_PCT + 0.4010 * FT_PCT + 
(possession stats) 0.0510 * STL_PG - 0.0480 * TOV_PG + 0.0470 OREB_PG = +0.0470 + 0.0366 * DREB_PG - 0.0313 BLKA_PG
```

The playoff formula is fundamentally the same, but the strength of each factor changes. Shooting efficiency still matters the most,  but it doesn't dominate quite as much. 

```
Win probability (%) = 
(shooting stats) 2.1975 * FG_PCT + 0.8077 * FG3_PCT + 0.3102 * FT_PCT + 
(possession stats) 0.0544 * STL_PG - 0.0417 * TOV_PG + 0.0363 * DREB_PB + 0.0296 * OREB_PB
```